# P10.6-AI — Notebook 56: evaluación interna y exportación final

Evalúa el mejor checkpoint del Notebook 55 sobre el `internal_test` reservado por el Notebook 54. Calcula métricas globales, por clase y por nivel; registra falsos negativos severos y exporta el artefacto final solo si supera los gates mínimos.

`humanReviewRequired=true` · `notClinicalDiagnosis=true` · `officialTestAccessed=false`


In [ ]:
# 1) Dependencias mínimas
from __future__ import annotations
import importlib.util
import subprocess
import sys

packages = {
    "pydicom": "pydicom",
    "timm": "timm",
    "tqdm": "tqdm",
    "sklearn": "scikit-learn",
}
missing = [pkg for mod, pkg in packages.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])


In [ ]:
# 2) GPU y Google Drive
import json
from pathlib import Path

import torch
from google.colab import drive  # type: ignore

if not torch.cuda.is_available():
    raise RuntimeError("Seleccioná GPU T4 o superior en el runtime de Colab.")

DEVICE = torch.device("cuda")
print({"gpu": torch.cuda.get_device_name(0), "torch": torch.__version__})
drive.mount("/content/drive", force_remount=False)


In [ ]:
# 3) Clonar o actualizar la rama
REPO_URL = "https://github.com/EnzoAA004/PFI_MVPTest_Enzo_AImodule.git"
REPO_ROOT = Path("/content/PFI_MVPTest_Enzo_AImodule")
REPO_REF = "enzo/p10-6-ai-rsna-findings"

if not (REPO_ROOT / ".git").exists():
    subprocess.check_call([
        "git", "clone", "--branch", REPO_REF, "--single-branch",
        REPO_URL, str(REPO_ROOT),
    ])
else:
    subprocess.check_call(["git", "fetch", "origin"], cwd=REPO_ROOT)
    subprocess.check_call(["git", "checkout", REPO_REF], cwd=REPO_ROOT)
    subprocess.check_call(["git", "pull", "--ff-only"], cwd=REPO_ROOT)

sys.path.insert(0, str(REPO_ROOT / "ai_service"))

from pfi_ai_service.training.rsna_central_training import (
    TrainConfig,
    build_cache,
    ensure_local_subset,
)
from pfi_ai_service.training.rsna_central_evaluation import (
    attach_coordinates_safe,
    evaluate,
    export_final_artifact,
    load_checkpoint,
    load_internal_test_manifest,
    sha256_file,
)

REPO_SHA = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_ROOT,
    text=True,
).strip()
print({"repoSha": REPO_SHA})


In [ ]:
# 4) Rutas y configuración
PFI_ROOT = Path("/content/drive/MyDrive/PFI_MVP")
SPLIT_ROOT = PFI_ROOT / "results" / "P10_6_rsna_findings" / "notebook54_split"
TRAINING_ROOT = PFI_ROOT / "results" / "P10_6_rsna_findings" / "notebook55_training"
EVALUATION_ROOT = PFI_ROOT / "results" / "P10_6_rsna_findings" / "notebook56_evaluation"
MODEL_ROOT = PFI_ROOT / "models" / "P10_6_rsna_findings" / "central_stenosis_sagittal_t2_2p5d"
CHECKPOINT_PATH = MODEL_ROOT / "checkpoints" / "best_checkpoint.pt"
FINAL_MODEL_PATH = MODEL_ROOT / "rsna_central_stenosis_sagittal_t2_2p5d.pt"

LOCAL_ROOT = Path("/content/RSNA_LUMBAR_DISC")
CACHE_ROOT = Path("/content/rsna_central_stenosis_cache")
COMPETITION = "rsna-2024-lumbar-spine-degenerative-classification"

EVALUATION_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

GATES = {
    "macro_f1": 0.45,
    "balanced_accuracy": 0.50,
    "severe_recall": 0.50,
}
print({
    "checkpoint": str(CHECKPOINT_PATH),
    "evaluationRoot": str(EVALUATION_ROOT),
    "finalModel": str(FINAL_MODEL_PATH),
    "gates": GATES,
})


In [ ]:
# 5) Abrir por primera vez el internal test y verificar separación
internal_manifest, split_summary = load_internal_test_manifest(SPLIT_ROOT)

train_manifest = __import__("pandas").read_csv(
    SPLIT_ROOT / "train_manifest.csv",
    dtype={"study_id": str, "series_id": str},
)
validation_manifest = __import__("pandas").read_csv(
    SPLIT_ROOT / "validation_manifest.csv",
    dtype={"study_id": str, "series_id": str},
)

internal_ids = set(internal_manifest["study_id"])
train_ids = set(train_manifest["study_id"])
validation_ids = set(validation_manifest["study_id"])

overlaps = {
    "train_internal_test": len(train_ids & internal_ids),
    "validation_internal_test": len(validation_ids & internal_ids),
}
if any(overlaps.values()):
    raise RuntimeError(f"Se detectó fuga de estudios: {overlaps}")

print({
    "internalTestRows": int(len(internal_manifest)),
    "internalTestStudies": int(internal_manifest["study_id"].nunique()),
    "overlaps": overlaps,
    "officialTestAccessed": False,
})


In [ ]:
# 6) Preparar localmente solo las series del internal test
# El Notebook 55 ya puede haber dejado el CSV y parte del dataset en /content.
# Si faltan series, se solicita el token de Kaggle sin mostrarlo.

import getpass
import pandas as pd

token = ""
required_series = set(internal_manifest["series_id"].astype(str))
local_series = LOCAL_ROOT / "train_images"

def internal_subset_ready() -> bool:
    if not (LOCAL_ROOT / "train_label_coordinates.csv").is_file():
        return False
    if not local_series.is_dir():
        return False
    for series_id in list(required_series)[:50]:
        matches = list(local_series.glob(f"*/{series_id}"))
        if not matches or not any(matches[0].glob("*.dcm")):
            return False
    return True

if not internal_subset_ready():
    token = getpass.getpass("Pegá tu KAGGLE_API_TOKEN (no se mostrará): " ).strip()
    if not token:
        raise RuntimeError("No se ingresó token de Kaggle.")

empty_manifest = internal_manifest.iloc[0:0].copy()
ensure_local_subset(
    internal_manifest,
    empty_manifest,
    LOCAL_ROOT,
    COMPETITION,
    token,
)
token = ""
print({"localRoot": str(LOCAL_ROOT), "internalSubsetReady": True})


In [ ]:
# 7) Unir coordenadas y construir cache 2.5D del internal test
missing_report = EVALUATION_ROOT / "missing_coordinate_samples.csv"
internal_samples = attach_coordinates_safe(
    internal_manifest,
    LOCAL_ROOT,
    missing_report,
    max_missing_rate=0.02,
)

checkpoint_preview = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)
cfg_dict = dict(checkpoint_preview["config"])
cfg_dict["num_workers"] = 0
CFG = TrainConfig(**cfg_dict)

build_cache(
    internal_samples,
    CACHE_ROOT,
    "internal_test",
    CFG,
)

print({
    "internalSamples": int(len(internal_samples)),
    "internalStudies": int(internal_samples["study_id"].nunique()),
    "classDistribution": internal_samples["severity"].value_counts().to_dict(),
    "numWorkers": CFG.num_workers,
})


In [ ]:
# 8) Cargar el mejor checkpoint y ejecutar evaluación
model, checkpoint = load_checkpoint(CHECKPOINT_PATH, DEVICE)

metrics = evaluate(
    model,
    internal_samples,
    CACHE_ROOT,
    EVALUATION_ROOT,
    CFG,
    DEVICE,
)

print(json.dumps(metrics, indent=2))


In [ ]:
# 9) Aplicar gates y exportar el modelo final
gate_results = {
    "macroF1": metrics["macro_f1"] >= GATES["macro_f1"],
    "balancedAccuracy": (
        metrics["balanced_accuracy"] >= GATES["balanced_accuracy"]
    ),
    "severeRecall": metrics["severe_recall"] >= GATES["severe_recall"],
    "noStudyLeakage": not any(overlaps.values()),
    "officialTestAccessed": False,
    "humanReviewRequired": True,
    "notClinicalDiagnosis": True,
}

approved = (
    gate_results["macroF1"]
    and gate_results["balancedAccuracy"]
    and gate_results["severeRecall"]
    and gate_results["noStudyLeakage"]
    and gate_results["officialTestAccessed"] is False
    and gate_results["humanReviewRequired"]
    and gate_results["notClinicalDiagnosis"]
)

output_names = [
    "evaluation_metrics.json",
    "internal_test_predictions.csv",
    "metrics_by_class.csv",
    "metrics_by_level.csv",
    "confusion_matrix.csv",
    "severe_false_negatives.csv",
    "calibration_curve.csv",
    "missing_coordinate_samples.csv",
]
evaluation_hashes = {
    name: sha256_file(EVALUATION_ROOT / name)
    for name in output_names
    if (EVALUATION_ROOT / name).is_file()
}

exported_path = export_final_artifact(
    CHECKPOINT_PATH,
    FINAL_MODEL_PATH,
    metrics,
    evaluation_hashes,
    approved,
)

evaluation_summary = {
    "schemaVersion": "pfi.rsna-central-stenosis-evaluation.v1",
    "repoSha": REPO_SHA,
    "checkpointSha256": sha256_file(CHECKPOINT_PATH),
    "internalTestManifestSha256": sha256_file(
        SPLIT_ROOT / "internal_test_manifest.csv"
    ),
    "metrics": metrics,
    "gates": GATES,
    "gateResults": gate_results,
    "approved": approved,
    "finalArtifact": str(exported_path) if exported_path else None,
    "finalArtifactSha256": (
        sha256_file(exported_path) if exported_path else None
    ),
    "outputHashes": evaluation_hashes,
    "governance": {
        "humanReviewRequired": True,
        "notClinicalDiagnosis": True,
        "predictionStatus": "pending_review",
        "officialTestAccessed": False,
    },
}

summary_path = EVALUATION_ROOT / "evaluation_summary.json"
with summary_path.open("w", encoding="utf-8") as handle:
    json.dump(evaluation_summary, handle, indent=2, ensure_ascii=False)

print(json.dumps(gate_results, indent=2))
print({
    "status": (
        "APPROVED_FINAL_MODEL"
        if approved
        else "BASELINE_RETAINED_NOT_APPROVED"
    ),
    "finalArtifact": str(exported_path) if exported_path else None,
    "evaluationSummary": str(summary_path),
})


In [ ]:
# 10) Gate final de cierre
required_outputs = [
    EVALUATION_ROOT / "evaluation_metrics.json",
    EVALUATION_ROOT / "internal_test_predictions.csv",
    EVALUATION_ROOT / "metrics_by_class.csv",
    EVALUATION_ROOT / "metrics_by_level.csv",
    EVALUATION_ROOT / "confusion_matrix.csv",
    EVALUATION_ROOT / "severe_false_negatives.csv",
    EVALUATION_ROOT / "calibration_curve.csv",
    EVALUATION_ROOT / "evaluation_summary.json",
]

missing_outputs = [str(path) for path in required_outputs if not path.is_file()]
if missing_outputs:
    raise RuntimeError("Faltan outputs de evaluación:\n- " + "\n- ".join(missing_outputs))

if approved:
    if not FINAL_MODEL_PATH.is_file():
        raise RuntimeError("El modelo fue aprobado pero no se exportó.")
    print({
        "status": "P10_6_RSNA_CENTRAL_STENOSIS_CLOSED",
        "modelApproved": True,
        "finalArtifact": str(FINAL_MODEL_PATH),
        "humanReviewRequired": True,
        "notClinicalDiagnosis": True,
    })
else:
    print({
        "status": "P10_6_RSNA_CENTRAL_STENOSIS_BASELINE_ONLY",
        "modelApproved": False,
        "reason": "No se superaron todos los gates mínimos.",
        "humanReviewRequired": True,
        "notClinicalDiagnosis": True,
    })
